# Entity-Based Study Similarity Pipeline

## Goal

Given two clinical studies, produce a comprehensive similarity score with a breakdown across entity usage, section structure, deviation from the corpus, protocol text, and metadata.

## Notebook Flow

1. **Setup & Preparation** (Section 0)
2. **Entity Type Similarity** (Section 1.1)
3. **Section Matching** (Section 1.2)
4. **Entity Deviation Analysis** (Section 1.3)
5. **Protocol Embeddings** (Section 1.4)
6. **Study Metadata Comparison** (Section 1.5)
7. **Final Aggregation**


## Section 0: Setup & Preparation

In [1]:
import sys
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pandas as pd
from src.similarity import (
    graph_summary, study_preview, protocol_studies, protocol_csr_studies,
    build_entity_counts_matrix, entity_type_overlap, protocol_sections, map_section_lists,
    section_entity_matrix, global_entity_stats, significant_deviations,
    nonentity_section_counts, protocol_embedding_frame, study_protocol_embedding,
    study_metadata_frame, compare_studies, METADATA_FIELDS,
)
from src.components.section_distribution import section_distribution_similarity
from src.components.protocol_embedding import protocol_embedding_similarity
from src.components.metadata import metadata_similarity
names = protocol_studies()["study_name"].tolist()
study_pairs = list(zip(names[0::2], names[1::2]))[:6]
print(graph_summary())
display(study_preview())


[public-ctg] Total nodes: 6160, Total relationships: 11697


,study_title,comparative_study,subject_type,study_phase,study_status
0,"MERIT-2 : Long Term, Multicenter, Single-arm, ...",N,PATIENTS,PHASE2,COMPLETED
1,Improve Sudden Cardiac Arrest Study,N,PATIENTS,NA,COMPLETED
2,Fixed Dose Intervention Trial of New England E...,Y,PATIENTS,PHASE4,COMPLETED
3,Improving Advanced Cancer Patient-Centered Car...,Y,PATIENTS,NA,COMPLETED
4,"An Open Label, Multicenter Study to Assess the...",N,PATIENTS,PHASE4,COMPLETED


### Study Filtering

Identify studies with PROTOCOL documents and studies with both PROTOCOL and CSR documents.

In [2]:

protocol_studies_result = protocol_studies()
protocol_study_list = protocol_studies_result["study_name"].tolist()
print("All studies with PROTOCOL documents loaded:")
display(protocol_studies_result)
protocol_csr_studies_result = protocol_csr_studies()
print("All studies with PROTOCOL & CSR documents loaded:")
display(protocol_csr_studies_result)


All studies with PROTOCOL documents loaded:


,study_name,study_node_id
0,NCT02060721,4:3cf6aee3-b618-46ae-9144-2e0d3ee5ddf7:5717
1,NCT02099721,4:3cf6aee3-b618-46ae-9144-2e0d3ee5ddf7:80
2,NCT02188121,4:3cf6aee3-b618-46ae-9144-2e0d3ee5ddf7:284
3,NCT02374255,4:3cf6aee3-b618-46ae-9144-2e0d3ee5ddf7:453
4,NCT02427958,4:3cf6aee3-b618-46ae-9144-2e0d3ee5ddf7:473
5,NCT02505984,4:3cf6aee3-b618-46ae-9144-2e0d3ee5ddf7:675
6,NCT02528357,4:3cf6aee3-b618-46ae-9144-2e0d3ee5ddf7:744
7,NCT02564926,4:3cf6aee3-b618-46ae-9144-2e0d3ee5ddf7:993
8,NCT02709655,4:3cf6aee3-b618-46ae-9144-2e0d3ee5ddf7:1164
9,NCT02713867,4:3cf6aee3-b618-46ae-9144-2e0d3ee5ddf7:1414


All studies with PROTOCOL & CSR documents loaded:


,study_name,study_node_id
0,NCT02060721,4:3cf6aee3-b618-46ae-9144-2e0d3ee5ddf7:5717
1,NCT02099721,4:3cf6aee3-b618-46ae-9144-2e0d3ee5ddf7:80
2,NCT02188121,4:3cf6aee3-b618-46ae-9144-2e0d3ee5ddf7:284
3,NCT02374255,4:3cf6aee3-b618-46ae-9144-2e0d3ee5ddf7:453
4,NCT02427958,4:3cf6aee3-b618-46ae-9144-2e0d3ee5ddf7:473
5,NCT02505984,4:3cf6aee3-b618-46ae-9144-2e0d3ee5ddf7:675
6,NCT02528357,4:3cf6aee3-b618-46ae-9144-2e0d3ee5ddf7:744
7,NCT02564926,4:3cf6aee3-b618-46ae-9144-2e0d3ee5ddf7:993
8,NCT02709655,4:3cf6aee3-b618-46ae-9144-2e0d3ee5ddf7:1164
9,NCT02713867,4:3cf6aee3-b618-46ae-9144-2e0d3ee5ddf7:1414


## Section 1.1: Entity Type Similarity

**Goal**: Build a study × entity count matrix from standardized entities extracted from protocol section content.

In [3]:

entity_counts_matrix = build_entity_counts_matrix(protocol_study_list)
display(entity_counts_matrix)


entity_col,ASSESSMENT- 1 hour blood collection,ASSESSMENT- 12-Lead ECG,ASSESSMENT- 24-hour diet recall,ASSESSMENT- 6-minute walk test,ASSESSMENT- 6MWT,ASSESSMENT- AARP assessment,ASSESSMENT- AE/Toxicity assessments,ASSESSMENT- AEs/SAEs,ASSESSMENT- Acceptability rating of mobile phone,ASSESSMENT- Addressing key barriers,...,OBJECTIVE- Investigate trauma effects on memory,OBJECTIVE- Measure program effectiveness and implementation,OBJECTIVE- Primary objective,OBJECTIVE- Reduce PPD and PP-PTSD symptoms,"OBJECTIVE- Reduction of total body FM, BW, abdominal VAT and SAT volume",OBJECTIVE- Renal function requirement,OBJECTIVE- Report SAEs associated with overdose,OBJECTIVE- Sample size estimation,OBJECTIVE- Understand efficacy and safety of dosing regimen,OBJECTIVE- Widespread data distribution
study_id,,,,,,,,,,,,,,,,,,,,,
NCT02060721,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
NCT02099721,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1
NCT02188121,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
NCT02374255,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
NCT02427958,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
NCT02505984,0,0,0,0,0,0,0,0,0,0,...,1,0,0,1,0,0,0,0,0,0
NCT02528357,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
NCT02564926,0,3,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
NCT02709655,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Understanding the Entity Counts Matrix

Rows are studies. Columns are `entity_type- entity_name`. Each cell is how many extracted mentions map to that canonical entity.

### Entity Type Similarity (1.1) — Demo Pairs

Cosine on per-study proportions, plus shared and unique entity counts.

In [4]:

rows = []
for study_a, study_b in study_pairs:
    overlap = entity_type_overlap(entity_counts_matrix, study_a, study_b)
    rows.append({
        "study_a": study_a,
        "study_b": study_b,
        "entity_type_similarity": overlap["similarity"],
        "num_shared_types": overlap["num_shared_types"],
        "num_unique_a": overlap["num_unique_a"],
        "num_unique_b": overlap["num_unique_b"],
    })
entity_type_sim_df = pd.DataFrame(rows).sort_values("entity_type_similarity", ascending=False)
display(entity_type_sim_df)


,study_a,study_b,entity_type_similarity,num_shared_types,num_unique_a,num_unique_b
1,NCT02188121,NCT02374255,0.685597,1,33,34
3,NCT02528357,NCT02564926,0.480255,3,55,45
5,NCT02725515,NCT02752048,0.030308,1,42,48
0,NCT02060721,NCT02099721,0.000000,0,24,53
2,NCT02427958,NCT02505984,0.000000,0,33,33
4,NCT02709655,NCT02713867,0.000000,0,32,39


## Section 1.2: Section Matching Similarity

Map functionally identical sections across studies with local embeddings. Parent sections are aligned first, then child sections, one to one.

In [5]:

semantic_mappings = []
for study_a, study_b in study_pairs:
    mapping = map_section_lists(protocol_sections(study_a), protocol_sections(study_b))
    semantic_mappings.append(mapping)
    print(study_a, study_b, "mapped", len(mapping))
display(pd.DataFrame([{
    "section_a": left.name, "section_b": right.name, "cosine": score
} for left, right, score in semantic_mappings[0]]).head(12))


NCT02060721 NCT02099721 mapped 8


NCT02188121 NCT02374255 mapped 1


NCT02427958 NCT02505984 mapped 16


NCT02528357 NCT02564926 mapped 31


NCT02709655 NCT02713867 mapped 31


NCT02725515 NCT02752048 mapped 27


,section_a,section_b,cosine
0,0 Eligibility Criteria,0 Eligibility Criteria,1.000000
1,1 Written informed consent to participate in t...,23 DEC 2013 Clinical Investigation Plan,0.435241
2,1 Permanent discontinuation of DB study treatm...,23 DEC 2013 Clinical Investigation Plan,0.430934
3,2 STUDY OBJECTIVES,23 DEC 2013 Clinical Investigation Plan,0.426080
4,1.3 Purpose and rationale of the study,1.1 Study Purpose,0.730342
5,5.1.7 Compliance with study treatment,3.4 Changes to the Study System,0.540313
6,11.3.2 Analysis of subject disposition and study,1.2 Study Description and Scope,0.532019
7,5.1.4 Treatment assignment,1.0 Prevention Subject,0.495598


### LLM-Based Mapping Quality Review

Optional. Left off unless `RUN_LLM_REVIEW` is set. The chat connector is the same factory used for entity extraction.

In [6]:

RUN_LLM_REVIEW = False
if RUN_LLM_REVIEW:
    from src.extraction import build_chat_model
    print(type(build_chat_model()).__name__)
else:
    print("LLM mapping review skipped")


LLM mapping review skipped


### Section-Level Entity Count Aggregation

In [7]:

for index, ((study_a, study_b), mapping) in enumerate(zip(study_pairs, semantic_mappings)):
    section_count_matrix_filtered = section_entity_matrix([study_a, study_b], mapping)
    if index < 2:
        display(section_count_matrix_filtered)


,0 Eligibility Criteria -> 0 Eligibility Criteria,1 Written informed consent to participate in the study -> 23 DEC 2013 Clinical Investigation Plan,1 Permanent discontinuation of DB study treatment due to -> 23 DEC 2013 Clinical Investigation Plan,2 STUDY OBJECTIVES -> 23 DEC 2013 Clinical Investigation Plan,1.3 Purpose and rationale of the study -> 1.1 Study Purpose,5.1.7 Compliance with study treatment -> 3.4 Changes to the Study System,11.3.2 Analysis of subject disposition and study -> 1.2 Study Description and Scope,5.1.4 Treatment assignment -> 1.0 Prevention Subject
NCT02060721,5.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0
NCT02099721,2.0,0.0,0.0,8.0,0.0,0.0,8.0,0.0


,0 Eligibility Criteria -> 0 Eligibility Criteria
NCT02188121,3.0
NCT02374255,5.0


### Section Distribution Similarity (1.2) — Demo Pairs

In [8]:

section_rows = []
for (study_a, study_b), mapping in zip(study_pairs, semantic_mappings):
    frame = section_entity_matrix([study_a, study_b], mapping)
    score = section_distribution_similarity(frame.loc[study_a].to_numpy(), frame.loc[study_b].to_numpy()) if len(frame.columns) else 0.0
    section_rows.append({"study_a": study_a, "study_b": study_b, "section_distribution_similarity": score, "mapped_sections": len(frame.columns)})
section_dist_sim_df = pd.DataFrame(section_rows).sort_values("section_distribution_similarity", ascending=False)
display(section_dist_sim_df)


,study_a,study_b,section_distribution_similarity,mapped_sections
1,NCT02188121,NCT02374255,1.000000,1
2,NCT02427958,NCT02505984,0.469776,16
0,NCT02060721,NCT02099721,0.420230,8
5,NCT02725515,NCT02752048,0.267466,27
3,NCT02528357,NCT02564926,0.259554,31
4,NCT02709655,NCT02713867,0.174442,31


## Section 1.3: Entity Deviation Analysis

In [9]:

global_entity_stats_df = global_entity_stats(entity_counts_matrix)
display(global_entity_stats_df.head(12))


,mean,std,studies_present
entity_col,,,
ASSESSMENT- 1 hour blood collection,0.04,0.200000,1
ASSESSMENT- 12-Lead ECG,0.36,0.994987,3
ASSESSMENT- 24-hour diet recall,0.24,0.830662,2
ASSESSMENT- 6-minute walk test,0.08,0.400000,1
ASSESSMENT- 6MWT,0.04,0.200000,1
ASSESSMENT- AARP assessment,0.04,0.200000,1
ASSESSMENT- AE/Toxicity assessments,0.04,0.200000,1
ASSESSMENT- AEs/SAEs,0.04,0.200000,1
ASSESSMENT- Acceptability rating of mobile phone,0.04,0.200000,1


In [10]:

pair_to_significant_deviations = {}
for study_a, study_b in study_pairs:
    pair_to_significant_deviations[(study_a, study_b)] = significant_deviations(entity_counts_matrix, study_a, study_b)
for index, pair in enumerate(study_pairs):
    display(pair_to_significant_deviations[pair].head(8))
    if index == 0:
        break


,entity,mu,count_a,count_b,delta_a,delta_b,per_entity_sim,weight
448,INCLUSION_CRITERION- Age 5 years or older,8.84,17.0,0.0,8.16,-8.84,0.0,8.84
661,OBJECTIVE- Assess long-term safety,1.80,9.0,0.0,7.20,-1.80,0.0,7.20
560,INCLUSION_CRITERION- Written informed consent,2.52,9.0,0.0,6.48,-2.52,0.0,6.48
387,EXCLUSION_CRITERION- Pregnancy and contraception,5.20,0.0,0.0,-5.20,-5.20,1.0,5.20
535,INCLUSION_CRITERION- Pregnancy and contraception,1.20,6.0,0.0,4.80,-1.20,0.0,4.80
183,ASSESSMENT- Vital Signs,0.60,5.0,0.0,4.40,-0.60,0.0,4.40
450,INCLUSION_CRITERION- Age requirement,3.84,0.0,0.0,-3.84,-3.84,1.0,3.84
187,ASSESSMENT- Written informed consent,0.36,0.0,3.0,-0.36,2.64,0.0,2.64


### Section-Level Deviation Localization

In [11]:

first_a, first_b = study_pairs[0]
first_mapping = semantic_mappings[0]
level2 = section_entity_matrix([first_a, first_b], first_mapping)
display(pd.concat([
    level2.loc[[first_a]].T.rename(columns={first_a: "study_a_count"}),
    level2.loc[[first_b]].T.rename(columns={first_b: "study_b_count"}),
], axis=1).head(12))


,study_a_count,study_b_count
0 Eligibility Criteria -> 0 Eligibility Criteria,5.0,2.0
1 Written informed consent to participate in the study -> 23 DEC 2013 Clinical Investigation Plan,0.0,0.0
1 Permanent discontinuation of DB study treatment due to -> 23 DEC 2013 Clinical Investigation Plan,0.0,0.0
2 STUDY OBJECTIVES -> 23 DEC 2013 Clinical Investigation Plan,2.0,8.0
1.3 Purpose and rationale of the study -> 1.1 Study Purpose,0.0,0.0
5.1.7 Compliance with study treatment -> 3.4 Changes to the Study System,0.0,0.0
11.3.2 Analysis of subject disposition and study -> 1.2 Study Description and Scope,0.0,8.0
5.1.4 Treatment assignment -> 1.0 Prevention Subject,0.0,0.0


### 1.3 Additional Demo: Single Study vs All Studies

In [12]:

target_study = protocol_study_list[0]
similarity_rows = []
for other in protocol_study_list:
    if other == target_study:
        continue
    overlap = entity_type_overlap(entity_counts_matrix, target_study, other)
    similarity_rows.append({"other_study": other, "entity_type_similarity": overlap["similarity"]})
similarity_df = pd.DataFrame(similarity_rows).sort_values("entity_type_similarity", ascending=False)
display(similarity_df)


,other_study,entity_type_similarity
3,NCT02427958,0.734026
17,NCT03461016,0.696078
1,NCT02188121,0.682398
13,NCT02879578,0.678359
5,NCT02528357,0.670883
22,NCT04246593,0.661906
10,NCT02752048,0.638460
7,NCT02709655,0.606087
18,NCT03542149,0.552219
2,NCT02374255,0.540698


In [13]:

most = similarity_df.iloc[0]
least = similarity_df.iloc[-1]
for label, row in (("most", most), ("least", least)):
    study_df = significant_deviations(entity_counts_matrix, target_study, row["other_study"]).head(8)
    print(label, target_study, row["other_study"])
    display(study_df)


most NCT02060721 NCT02427958


,entity,mu,count_a,count_b,delta_a,delta_b,per_entity_sim,weight
448,INCLUSION_CRITERION- Age 5 years or older,8.84,17.0,17.0,8.16,8.16,1.0,8.16
387,EXCLUSION_CRITERION- Pregnancy and contraception,5.20,0.0,13.0,-5.20,7.80,0.0,7.80
661,OBJECTIVE- Assess long-term safety,1.80,9.0,9.0,7.20,7.20,1.0,7.20
560,INCLUSION_CRITERION- Written informed consent,2.52,9.0,9.0,6.48,6.48,1.0,6.48
535,INCLUSION_CRITERION- Pregnancy and contraception,1.20,6.0,0.0,4.80,-1.20,0.0,4.80
183,ASSESSMENT- Vital Signs,0.60,5.0,0.0,4.40,-0.60,0.0,4.40
610,INTERVENTION- Leuprorelin 3.75 mg every 4 weeks,0.16,0.0,4.0,-0.16,3.84,0.0,3.84
450,INCLUSION_CRITERION- Age requirement,3.84,0.0,0.0,-3.84,-3.84,1.0,3.84


least NCT02060721 NCT04321252


,entity,mu,count_a,count_b,delta_a,delta_b,per_entity_sim,weight
448,INCLUSION_CRITERION- Age 5 years or older,8.84,17.0,0.0,8.16,-8.84,0.0,8.84
450,INCLUSION_CRITERION- Age requirement,3.84,0.0,12.0,-3.84,8.16,0.0,8.16
387,EXCLUSION_CRITERION- Pregnancy and contraception,5.20,0.0,13.0,-5.20,7.80,0.0,7.80
661,OBJECTIVE- Assess long-term safety,1.80,9.0,0.0,7.20,-1.80,0.0,7.20
560,INCLUSION_CRITERION- Written informed consent,2.52,9.0,0.0,6.48,-2.52,0.0,6.48
535,INCLUSION_CRITERION- Pregnancy and contraception,1.20,6.0,0.0,4.80,-1.20,0.0,4.80
136,ASSESSMENT- Pregnancy and contraception,0.40,0.0,5.0,-0.40,4.60,0.0,4.60
183,ASSESSMENT- Vital Signs,0.60,5.0,0.0,4.40,-0.60,0.0,4.40


## Section 1.4: Protocol Document Embedding Creation (non-entity content)

In [14]:

per_study_counts = nonentity_section_counts(protocol_study_list)
display(per_study_counts)


,study_id,non_entity_sections
0,NCT02060721,74
1,NCT02099721,73
2,NCT02188121,73
3,NCT02427958,73
4,NCT02505984,23
5,NCT02528357,73
6,NCT02564926,73
7,NCT02709655,73
8,NCT02713867,73
9,NCT02725515,73


In [15]:

protocol_embed_df = protocol_embedding_frame(protocol_study_list)
display(protocol_embed_df.head())


C:\Users\filipp03\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\models\bert\modeling_bert.py:413: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


,study_id,dimensions,norm
0,NCT02060721,384,1.0
1,NCT02099721,384,1.0
2,NCT02188121,384,1.0
3,NCT02374255,0,0.0
4,NCT02427958,384,1.0


### Protocol Embedding Similarity (1.4) — Demo Pairs

In [16]:

embed_rows = []
for study_a, study_b in study_pairs:
    vector_a = study_protocol_embedding(study_a)
    vector_b = study_protocol_embedding(study_b)
    score = protocol_embedding_similarity([vector_a] if vector_a.size else [], [vector_b] if vector_b.size else [])
    embed_rows.append({"study_a": study_a, "study_b": study_b, "protocol_embedding_similarity": score})
protocol_embedding_sim_df = pd.DataFrame(embed_rows).sort_values("protocol_embedding_similarity", ascending=False)
display(protocol_embedding_sim_df)


,study_a,study_b,protocol_embedding_similarity
4,NCT02709655,NCT02713867,0.520140
0,NCT02060721,NCT02099721,0.490610
5,NCT02725515,NCT02752048,0.406155
2,NCT02427958,NCT02505984,0.315240
3,NCT02528357,NCT02564926,0.245938
1,NCT02188121,NCT02374255,0.000000


## Section 1.5: Study Metadata Comparison

In [17]:

study_metadata_df = study_metadata_frame(protocol_study_list)
display(study_metadata_df.head(5))


,study_phase,indication,subject_type,design,blinding,comparative_study,allocation,sex,study_title,study_status
study_id,,,,,,,,,,
NCT02060721,PHASE2,Chronic Thromboembolic Pulmonary Hypertension,PATIENTS,SINGLE_GROUP,NONE,N,NA,ALL,"MERIT-2 : Long Term, Multicenter, Single-arm, ...",COMPLETED
NCT02099721,NA,Sudden Cardiac Arrest,PATIENTS,PARALLEL,NONE,N,NON_RANDOMIZED,ALL,Improve Sudden Cardiac Arrest Study,COMPLETED
NCT02188121,PHASE4,Serious Mental Illness,PATIENTS,PARALLEL,NONE,Y,RANDOMIZED,ALL,Fixed Dose Intervention Trial of New England E...,COMPLETED
NCT02374255,NA,Primary Stage IV Hepatobiliary,PATIENTS,PARALLEL,NONE,Y,RANDOMIZED,ALL,Improving Advanced Cancer Patient-Centered Car...,COMPLETED
NCT02427958,PHASE4,Central Precocious Puberty,PATIENTS,SINGLE_GROUP,NONE,N,NA,ALL,"An Open Label, Multicenter Study to Assess the...",COMPLETED


### Metadata Similarity Calculation

In [18]:

metadata_similarity_results = []
for study_a, study_b in study_pairs:
    score, _rows = metadata_similarity(study_metadata_df.loc[study_a].to_dict(), study_metadata_df.loc[study_b].to_dict(), METADATA_FIELDS)
    matched = sum(1 for row in _rows if row.get("match"))
    metadata_similarity_results.append({
        "study_a": study_a,
        "study_b": study_b,
        "metadata_similarity": score,
        "matched_count": matched,
        "total_fields": len(METADATA_FIELDS),
    })
metadata_sim_df = pd.DataFrame(metadata_similarity_results)
display(metadata_sim_df.sort_values("metadata_similarity", ascending=False))


,study_a,study_b,metadata_similarity,matched_count,total_fields
1,NCT02188121,NCT02374255,0.750,6,8
5,NCT02725515,NCT02752048,0.750,6,8
4,NCT02709655,NCT02713867,0.750,6,8
0,NCT02060721,NCT02099721,0.500,4,8
3,NCT02528357,NCT02564926,0.375,3,8
2,NCT02427958,NCT02505984,0.125,1,8


### Final Aggregation

Weighted sum of the five components. Defaults: entity type 0.30, section distribution 0.25, entity deviation 0.25, protocol embedding 0.15, metadata 0.05.

In [19]:

from src.engine import SimilarityWeights
results = [compare_studies(a, b, entity_counts_matrix, study_metadata_df, SimilarityWeights()) for a, b in study_pairs]
summary_df = pd.DataFrame([{
    "study_a": result.study_pair[0],
    "study_b": result.study_pair[1],
    "study_design_similarity": result.study_design_similarity,
    **{name: result.raw_scores[name] for name in ("entity_type", "section_distribution", "entity_deviation", "protocol_embedding", "metadata")},
} for result in results]).sort_values("study_design_similarity", ascending=False)
display(summary_df)


,study_a,study_b,study_design_similarity,entity_type,section_distribution,entity_deviation,protocol_embedding,metadata
1,NCT02188121,NCT02374255,0.586230,0.685597,1.000000,0.372204,0.000000,0.750
3,NCT02528357,NCT02564926,0.336223,0.480255,0.259554,0.286470,0.245938,0.375
0,NCT02060721,NCT02099721,0.280728,0.000000,0.420230,0.308316,0.490610,0.500
2,NCT02427958,NCT02505984,0.241407,0.000000,0.469776,0.281708,0.315240,0.125
5,NCT02725515,NCT02752048,0.238528,0.030308,0.267466,0.256583,0.406155,0.750
4,NCT02709655,NCT02713867,0.229310,0.000000,0.174442,0.280712,0.520140,0.750


In [20]:

ordered = sorted(results, key=lambda result: result.study_design_similarity)
for result in (ordered[-1], ordered[0]):
    print(result.study_pair, result.study_design_similarity)
    display(pd.DataFrame(result.artifacts["entity_comparison"]).query("status != 'absent'").head(10))
    display(pd.DataFrame(result.artifacts["section_mapping_table"]).head(10))
    display(pd.DataFrame(result.artifacts["deviation_level1_rows"]).head(10))
    display(pd.DataFrame(result.artifacts["deviation_level2_rows"]).head(10))
    display(pd.DataFrame(result.artifacts["section_score_rows"]).head(10))
    display(pd.DataFrame(result.artifacts["metadata_comparison"]))


('NCT02188121', 'NCT02374255') 0.5862299803244093


,entity,count_a,count_b,status
15,ASSESSMENT- Assessment of changes in goals of ...,0,1,unique_to_b
48,ASSESSMENT- Comparison of statin doses,1,0,unique_to_a
73,ASSESSMENT- Feasibility of GoC discussions,0,1,unique_to_b
74,ASSESSMENT- Feedback on training effectiveness,0,1,unique_to_b
88,ASSESSMENT- Hypertension outcomes,1,0,unique_to_a
98,ASSESSMENT- Measure adherence rates,1,0,unique_to_a
101,ASSESSMENT- Medication adherence rates,1,0,unique_to_a
106,ASSESSMENT- Meta-analysis of statin therapy,1,0,unique_to_a
130,ASSESSMENT- Patient satisfaction with goals of...,0,1,unique_to_b
159,ASSESSMENT- Screening for cardiovascular risk ...,1,0,unique_to_a


,section_a,section_b,total_mentions_a,total_mentions_b
0,0 Eligibility Criteria,0 Eligibility Criteria,3.0,5.0


,entity,mu,count_a,count_b,delta_a,delta_b,per_entity_sim,weight
0,ASSESSMENT- 1 hour blood collection,0.04,0.0,0.0,-0.04,-0.04,1.0,0.10
1,ASSESSMENT- 12-Lead ECG,0.36,0.0,0.0,-0.36,-0.36,1.0,0.36
2,ASSESSMENT- 24-hour diet recall,0.24,0.0,0.0,-0.24,-0.24,1.0,0.24
3,ASSESSMENT- 6-minute walk test,0.08,0.0,0.0,-0.08,-0.08,1.0,0.10
4,ASSESSMENT- 6MWT,0.04,0.0,0.0,-0.04,-0.04,1.0,0.10
5,ASSESSMENT- AARP assessment,0.04,0.0,0.0,-0.04,-0.04,1.0,0.10
6,ASSESSMENT- AE/Toxicity assessments,0.04,0.0,0.0,-0.04,-0.04,1.0,0.10
7,ASSESSMENT- AEs/SAEs,0.04,0.0,0.0,-0.04,-0.04,1.0,0.10
8,ASSESSMENT- Acceptability rating of mobile phone,0.04,0.0,0.0,-0.04,-0.04,1.0,0.10
9,ASSESSMENT- Addressing key barriers,0.04,0.0,0.0,-0.04,-0.04,1.0,0.10


,entity,section_index,delta_a_section,delta_b_section,agree_sign
0,INCLUSION_CRITERION- Advanced cancer patients,0,-1.0,1.0,False
1,INCLUSION_CRITERION- Life expectancy < 2 years,0,-0.5,0.5,False
2,INCLUSION_CRITERION- Ongoing treatment at stud...,0,0.5,-0.5,False
3,INCLUSION_CRITERION- Recent treatment with sta...,0,0.5,-0.5,False
4,INCLUSION_CRITERION- Speak English or Spanish,0,-0.5,0.5,False


...

,section_a,section_b,total_mentions_a,total_mentions_b
0,0 Eligibility Criteria,0 Eligibility Criteria,7.0,4.0
1,8 The patient has a history of severe drug all...,4 Allergies and Adverse Drug Reaction,0.0,0.0
2,19 The patient has or has had one or more of t...,2 Medical History and Concurrent Diseases,0.0,0.0
3,22 The patient has one or more clinical labora...,3 Physical and Laboratory Test Findings,0.0,0.0
4,3 The patient is able to understand the Inform...,1 Signed Written Informed Consent,0.0,0.0
5,"12 The patient, if a girl aged ≥10 years old, ...",3 Age and Reproductive Status,0.0,0.0
6,13 The patient has any other disorder for whic...,1 Target Disease Exceptions,0.0,0.0
7,1.2 Rationale for the Study,1.1 Study Rationale,0.0,0.0
8,3.1 Overview of the Study Design,3.1 Study Design and Duration,8.0,0.0
9,4.4 Independent Ethics Committees (IECs) and I...,2.2 Institutional Review Board/Independent Eth...,0.0,0.0


,entity,mu,count_a,count_b,delta_a,delta_b,per_entity_sim,weight
0,ASSESSMENT- 1 hour blood collection,0.04,0.0,0.0,-0.04,-0.04,1.0,0.10
1,ASSESSMENT- 12-Lead ECG,0.36,0.0,0.0,-0.36,-0.36,1.0,0.36
2,ASSESSMENT- 24-hour diet recall,0.24,0.0,0.0,-0.24,-0.24,1.0,0.24
3,ASSESSMENT- 6-minute walk test,0.08,0.0,0.0,-0.08,-0.08,1.0,0.10
4,ASSESSMENT- 6MWT,0.04,0.0,0.0,-0.04,-0.04,1.0,0.10
5,ASSESSMENT- AARP assessment,0.04,0.0,0.0,-0.04,-0.04,1.0,0.10
6,ASSESSMENT- AE/Toxicity assessments,0.04,0.0,0.0,-0.04,-0.04,1.0,0.10
7,ASSESSMENT- AEs/SAEs,0.04,0.0,0.0,-0.04,-0.04,1.0,0.10
8,ASSESSMENT- Acceptability rating of mobile phone,0.04,0.0,0.0,-0.04,-0.04,1.0,0.10
9,ASSESSMENT- Addressing key barriers,0.04,0.0,0.0,-0.04,-0.04,1.0,0.10


,entity,section_index,delta_a_section,delta_b_section,agree_sign
0,ASSESSMENT- Children’s Global Assessment Scale,13,1.0,-1.0,False
1,ASSESSMENT- Columbia-Suicide Severity Rating S...,28,0.5,-0.5,False
2,ASSESSMENT- Criteria for dose delay and discon...,26,-0.5,0.5,False
3,ASSESSMENT- Laboratory tests,28,0.5,-0.5,False
4,ASSESSMENT- Monitoring for infusion reactions,26,-0.5,0.5,False
5,ENDPOINT- Change in CDRS-R total score,13,0.5,-0.5,False
6,ENDPOINT- Patient safety and wellbeing,28,0.5,-0.5,False
7,EXCLUSION_CRITERION- Any Grade 4 drug-related AE,14,-1.0,1.0,False
8,EXCLUSION_CRITERION- Dosing interruption > 6 w...,14,-0.5,0.5,False
9,EXCLUSION_CRITERION- Grade 2 drug-related uvei...,14,-0.5,0.5,False


,section_a,section_b,cosine
0,0 Eligibility Criteria,0 Eligibility Criteria,1.000000
1,8 The patient has a history of severe drug all...,4 Allergies and Adverse Drug Reaction,0.741746
2,19 The patient has or has had one or more of t...,2 Medical History and Concurrent Diseases,0.472003
3,22 The patient has one or more clinical labora...,3 Physical and Laboratory Test Findings,0.471497
4,3 The patient is able to understand the Inform...,1 Signed Written Informed Consent,0.465430
5,"12 The patient, if a girl aged ≥10 years old, ...",3 Age and Reproductive Status,0.464890
6,13 The patient has any other disorder for whic...,1 Target Disease Exceptions,0.432518
7,1.2 Rationale for the Study,1.1 Study Rationale,0.894909
8,3.1 Overview of the Study Design,3.1 Study Design and Duration,0.801669
9,4.4 Independent Ethics Committees (IECs) and I...,2.2 Institutional Review Board/Independent Eth...,0.793459


,field,a,b,included,match
0,study_phase,PHASE3,PHASE3,True,True
1,indication,"Depressive Disorder, Major",Lung Cancer,True,False
2,subject_type,PATIENTS,PATIENTS,True,True
3,design,PARALLEL,PARALLEL,True,True
4,blinding,DOUBLE,NONE,True,False
5,comparative_study,Y,Y,True,True
6,allocation,RANDOMIZED,RANDOMIZED,True,True
7,sex,ALL,ALL,True,True
